# Интерактивный конспект: Автоэнкодеры (AE, VAE, CVAE, DAE)

Этот ноутбук — практическое сопровождение конспекта `SUMMARY.md`. Здесь все четыре варианта автоэнкодеров реализованы в **минимальной** форме на MNIST (28×28, серый), чтобы каждую модель можно было запустить за 1–3 минуты и руками потрогать ключевые эффекты:

1. **Vanilla AE** — реконструкция картинок через bottleneck.
2. **VAE** — probabilistic encoder + KL-регуляризация + sampling из $\mathcal{N}(0, I)$.
3. **CVAE** — добавляем класс как условие, генерируем «одну и ту же цифру в разных классах».
4. **Latent space visualization** — TSNE на VAE-латентах.
5. **Image retrieval** — KNN на VAE-латентах для поиска похожих цифр.

### Как использовать

- Запускайте ячейки по порядку.
- В разделах с заголовком **«Попробуй сам»** меняйте параметры (`LATENT_DIM`, `EPOCHS`, `NOISE_FACTOR`) и смотрите, как меняется поведение.
- Все модели обучаются на CPU или MPS/CUDA, если доступно. На M-серии Mac (MPS) полный ноутбук < 5 минут.

### Зависимости

`torch`, `torchvision`, `numpy`, `matplotlib`, `scikit-learn`. Все есть в venv `mfti`.


In [ ]:
import os
# PYTORCH_ENABLE_MPS_FALLBACK обязательно ДО импорта torch
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

# Воспроизводимость
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Выбор устройства
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print("device =", device, "| torch =", torch.__version__)

In [ ]:
# Загружаем MNIST (~10 МБ, скачается при первом запуске в mnist_data/)
BATCH = 128
train_set = datasets.MNIST(
    root="mnist_data/", train=True, transform=transforms.ToTensor(), download=True
)
test_set = datasets.MNIST(
    root="mnist_data/", train=False, transform=transforms.ToTensor(), download=True
)
train_loader = DataLoader(train_set, batch_size=BATCH, shuffle=True, drop_last=True)
test_loader = DataLoader(test_set, batch_size=BATCH, shuffle=False)
print(f"train: {len(train_set)} | test: {len(test_set)}")

# Вспомогательная: показать сетку картинок
def show_grid(images, titles=None, ncols=8, figsize=(12, 3)):
    images = images.detach().cpu().numpy()
    if images.ndim == 4:
        images = images.squeeze(1)  # (N, 1, 28, 28) -> (N, 28, 28)
    n = len(images)
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
    axes = np.array(axes).reshape(-1)
    for i in range(nrows * ncols):
        axes[i].axis("off")
        if i < n:
            axes[i].imshow(images[i], cmap="gray")
            if titles is not None:
                axes[i].set_title(str(titles[i]), fontsize=9)
    plt.tight_layout()
    plt.show()

## 1. Vanilla Autoencoder

**Идея.** Encoder $E: \mathbb{R}^{784} \to \mathbb{R}^d$ + Decoder $D: \mathbb{R}^d \to \mathbb{R}^{784}$. Loss — попиксельная MSE между входом и реконструкцией.

Здесь используем простую FC-архитектуру (без свёрток) — для MNIST 28×28 достаточно. На полноцветных лицах LFW из задания 21.1 уже понадобятся свёртки (`Conv2d` + `ConvTranspose2d`).

### Попробуй сам

- `LATENT_DIM = 16` — попробуй 4, 16, 64, 256. Чем меньше bottleneck, тем «грубее» реконструкция.
- `EPOCHS = 3` — больше эпох -> лучше реконструкция, но и время обучения растёт линейно.

In [ ]:
class VanillaAE(nn.Module):
    """Простой полносвязный AE 784 -> latent_dim -> 784."""

    def __init__(self, latent_dim: int = 16):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(784, 256), nn.ReLU(),
            nn.Linear(256, 64), nn.ReLU(),
            nn.Linear(64, latent_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64), nn.ReLU(),
            nn.Linear(64, 256), nn.ReLU(),
            nn.Linear(256, 784), nn.Sigmoid(),   # Sigmoid -> [0, 1]
        )

    def forward(self, x):
        x_flat = x.view(x.size(0), -1)
        z = self.encoder(x_flat)
        recon = self.decoder(z).view(-1, 1, 28, 28)
        return recon, z


# Параметры эксперимента (см. "Попробуй сам")
LATENT_DIM = 16
EPOCHS = 3

ae = VanillaAE(latent_dim=LATENT_DIM).to(device)
opt = torch.optim.Adam(ae.parameters(), lr=1e-3)
mse_loss = nn.MSELoss()

for epoch in range(1, EPOCHS + 1):
    ae.train()
    running = 0.0
    for xb, _ in train_loader:
        xb = xb.to(device)
        opt.zero_grad()
        recon, _ = ae(xb)
        loss = mse_loss(recon, xb)
        loss.backward()
        opt.step()
        running += loss.item() * xb.size(0)
    train_mse = running / len(train_set)
    print(f"AE Epoch {epoch}/{EPOCHS} | train MSE = {train_mse:.5f}")

# Визуализация: 8 пар (оригинал | реконструкция)
ae.eval()
with torch.no_grad():
    xb, _ = next(iter(test_loader))
    xb = xb[:8].to(device)
    recon, _ = ae(xb)
pairs = torch.cat([xb, recon], dim=0)
show_grid(pairs, ncols=8, figsize=(12, 3))
print("Верхний ряд — оригинал, нижний — реконструкция AE.")

## 2. Variational Autoencoder (VAE)

**Главное отличие от AE.** Encoder выдаёт не точку $z$, а распределение $q(z|x) = \mathcal{N}(\mu, \sigma^2)$. Loss = ELBO:

$$\mathcal{L}_{VAE} = D_{KL}(q(z|x) \| \mathcal{N}(0, I)) + \text{BCE}(x, \hat x)$$

**KL для двух гауссиан** (аналитика):

$$D_{KL} = -\frac{1}{2}\sum_i \bigl(1 + \log \sigma_i^2 - \mu_i^2 - \sigma_i^2\bigr)$$

Используем конвенцию `logsigma` $= \log\sigma^2$. Reparameterization: $z = \mu + \sigma \varepsilon$, $\sigma = \exp(0.5 \cdot \text{logsigma})$.

**Зачем KL.** Без него латентное пространство такое же неструктурированное, как у AE — sampling $z \sim \mathcal{N}(0, I)$ даёт мусор. KL-регуляризация заставляет $q(z|x)$ быть «похожим» на $\mathcal{N}(0, I)$ — и тогда sampling из априора работает.

### Попробуй сам

- `LATENT_DIM = 20` — на MNIST 20 достаточно, на лицах LFW обычно 32–64.
- `EPOCHS = 5` — после 5 эпох уже видны структурные кластеры в латенте.
- Поэкспериментируй с `kl_weight` (домножь KL на 0.1 или 10) и посмотри, как меняется качество семплов.

In [ ]:
class VAE(nn.Module):
    """Полносвязный VAE: 784 -> (mu, logsigma) -> z -> 784."""

    def __init__(self, latent_dim: int = 20):
        super().__init__()
        self.latent_dim = latent_dim
        self.fc_enc = nn.Linear(784, 400)
        self.fc_mu = nn.Linear(400, latent_dim)
        self.fc_logsigma = nn.Linear(400, latent_dim)
        self.fc_dec1 = nn.Linear(latent_dim, 400)
        self.fc_dec2 = nn.Linear(400, 784)

    def encode(self, x):
        h = F.relu(self.fc_enc(x.view(x.size(0), -1)))
        return self.fc_mu(h), self.fc_logsigma(h)

    def reparameterize(self, mu, logsigma):
        if self.training:
            std = torch.exp(0.5 * logsigma)
            eps = torch.randn_like(std)
            return mu + std * eps
        return mu

    def decode(self, z):
        h = F.relu(self.fc_dec1(z))
        return torch.sigmoid(self.fc_dec2(h)).view(-1, 1, 28, 28)

    def forward(self, x):
        mu, logsigma = self.encode(x)
        z = self.reparameterize(mu, logsigma)
        return self.decode(z), mu, logsigma


def kl_divergence(mu, logsigma):
    # D_KL(q || N(0,I)) для diag-гауссиан, sum по latent_dim и batch
    return -0.5 * torch.sum(1 + logsigma - mu.pow(2) - logsigma.exp())


def vae_loss(recon, x, mu, logsigma):
    bce = F.binary_cross_entropy(recon, x, reduction="sum")
    kl = kl_divergence(mu, logsigma)
    return bce + kl, bce, kl


# Обучение
LATENT_DIM_VAE = 20
EPOCHS_VAE = 5

vae = VAE(latent_dim=LATENT_DIM_VAE).to(device)
opt = torch.optim.Adam(vae.parameters(), lr=1e-3)

for epoch in range(1, EPOCHS_VAE + 1):
    vae.train()
    running = 0.0
    for xb, _ in train_loader:
        xb = xb.to(device)
        opt.zero_grad()
        recon, mu, logsigma = vae(xb)
        loss, bce, kl = vae_loss(recon, xb, mu, logsigma)
        loss.backward()
        opt.step()
        running += loss.item()
    train_elbo = running / len(train_set)
    print(f"VAE Epoch {epoch}/{EPOCHS_VAE} | train ELBO/img = {train_elbo:.3f}")

In [ ]:
# Реконструкции из test
vae.eval()
with torch.no_grad():
    xb, _ = next(iter(test_loader))
    xb = xb[:8].to(device)
    recon, _, _ = vae(xb)
print("VAE: верх — оригинал, низ — реконструкция")
show_grid(torch.cat([xb, recon], dim=0), ncols=8, figsize=(12, 3))

# Sampling из N(0, I) — работает благодаря KL!
with torch.no_grad():
    z = torch.randn(16, LATENT_DIM_VAE, device=device)
    samples = vae.decode(z)
print("VAE: 16 семплов из z ~ N(0, I) — узнаваемые цифры")
show_grid(samples, ncols=8, figsize=(12, 3))

## 3. Conditional VAE (CVAE)

**Идея.** Добавляем класс $y$ как условие. Encoder и Decoder получают `concat([вход, one_hot(y)])`:

$$q(z | x, y) = \mathcal{N}(\mu(x, y), \sigma^2(x, y))$$
$$p(x | z, y)$$

Латент $z$ теперь кодирует только **стиль** (как написана конкретная цифра), а класс задаётся снаружи. Главный демо-эффект — взять **один** $z$ и сгенерировать с ним десять разных цифр (0..9), меняя только `y`.

### Попробуй сам

- Меняй `z_fixed = torch.randn(1, LATENT_DIM_CVAE)` несколько раз — стиль будет разный, но 10 цифр в каждом сете будут «семейство похожего почерка».
- Уменьши `EPOCHS_CVAE` до 2 и посмотри, насколько хуже становится conditional sampling.

In [ ]:
class CVAE(nn.Module):
    """FC CVAE для MNIST. Условие — one-hot класса конкатенируется в encoder и decoder."""

    def __init__(self, num_classes: int = 10, latent_dim: int = 20):
        super().__init__()
        self.num_classes = num_classes
        self.latent_dim = latent_dim
        self.fc_enc = nn.Linear(784 + num_classes, 400)
        self.fc_mu = nn.Linear(400, latent_dim)
        self.fc_logsigma = nn.Linear(400, latent_dim)
        self.fc_dec1 = nn.Linear(latent_dim + num_classes, 400)
        self.fc_dec2 = nn.Linear(400, 784)

    def encode(self, x, y_oh):
        x_flat = x.view(x.size(0), -1)
        h = F.relu(self.fc_enc(torch.cat([x_flat, y_oh], dim=1)))
        return self.fc_mu(h), self.fc_logsigma(h)

    def reparameterize(self, mu, logsigma):
        if self.training:
            std = torch.exp(0.5 * logsigma)
            return mu + std * torch.randn_like(std)
        return mu

    def decode(self, z, y_oh):
        h = F.relu(self.fc_dec1(torch.cat([z, y_oh], dim=1)))
        return torch.sigmoid(self.fc_dec2(h)).view(-1, 1, 28, 28)

    def forward(self, x, y):
        y_oh = F.one_hot(y, self.num_classes).float()
        mu, logsigma = self.encode(x, y_oh)
        z = self.reparameterize(mu, logsigma)
        return self.decode(z, y_oh), mu, logsigma


LATENT_DIM_CVAE = 20
EPOCHS_CVAE = 5

cvae = CVAE(num_classes=10, latent_dim=LATENT_DIM_CVAE).to(device)
opt = torch.optim.Adam(cvae.parameters(), lr=1e-3)

for epoch in range(1, EPOCHS_CVAE + 1):
    cvae.train()
    running = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        recon, mu, logsigma = cvae(xb, yb)
        loss, _, _ = vae_loss(recon, xb, mu, logsigma)  # та же формула
        loss.backward()
        opt.step()
        running += loss.item()
    print(f"CVAE Epoch {epoch}/{EPOCHS_CVAE} | train ELBO/img = {running / len(train_set):.3f}")

In [ ]:
# Главный демо CVAE: один z, разные классы 0..9
cvae.eval()
with torch.no_grad():
    z_fixed = torch.randn(1, LATENT_DIM_CVAE, device=device).repeat(10, 1)  # (10, latent_dim)
    y_all = torch.arange(10, device=device)
    y_oh = F.one_hot(y_all, 10).float()
    samples = cvae.decode(z_fixed, y_oh)

print("CVAE: один z, классы 0..9 — десять цифр в одном 'стиле'")
show_grid(samples, ncols=10, figsize=(14, 1.8), titles=[str(k) for k in range(10)])

## 4. Latent space visualization (TSNE)

Возьмём $\mu$ из VAE-энкодера для test-набора и визуализируем 2D через TSNE. Раскрасим точки по классам — увидим, **выучил ли VAE классы неявно**.

Ожидаемый результат: **чёткие кластеры по цифрам**, потому что для эффективной реконструкции VAE вынужден разделить классы в латенте, даже без меток.

In [ ]:
from sklearn.manifold import TSNE

# Собираем латенты test через VAE encoder
vae.eval()
all_mu, all_y = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        mu, _ = vae.encode(xb.to(device))
        all_mu.append(mu.cpu().numpy())
        all_y.append(yb.numpy())
all_mu = np.concatenate(all_mu, axis=0)
all_y = np.concatenate(all_y, axis=0)

# Берём 3000 точек — TSNE быстрее, картина та же
N_TSNE = 3000
idx = np.random.RandomState(SEED).choice(len(all_mu), N_TSNE, replace=False)
emb = TSNE(n_components=2, perplexity=30, random_state=SEED, init="pca").fit_transform(all_mu[idx])

# Scatter
plt.figure(figsize=(7, 6))
sc = plt.scatter(emb[:, 0], emb[:, 1], c=all_y[idx], cmap="tab10", s=6, alpha=0.7)
plt.colorbar(sc, label="class")
plt.title(f"TSNE на VAE-латентах (n={N_TSNE}, perplexity=30)")
plt.xticks([]); plt.yticks([])
plt.show()
print("Видны кластеры по классам — VAE неявно выучил цифры.")

## 5. Image Retrieval через KNN на латентах

Энкодер обученного VAE/AE работает как **feature extractor**. На латентах можно построить KNN-индекс и искать «похожие картинки»:

1. Прогнать train через encoder -> база латентов.
2. `NearestNeighbors(algorithm='ball_tree')`.fit(latents)`.
3. Для query картинки получить $z$ и `kneighbors(z)`.

Раньше для этого в sklearn был `LSHForest`, но он **удалён в версии 0.21+**. Современная замена — `NearestNeighbors`.

In [ ]:
from sklearn.neighbors import NearestNeighbors

# Собираем латенты train (берём 10000 для скорости)
vae.eval()
train_subset_loader = DataLoader(train_set, batch_size=512, shuffle=False)
train_latents, train_images = [], []
with torch.no_grad():
    for xb, _ in train_subset_loader:
        mu, _ = vae.encode(xb.to(device))
        train_latents.append(mu.cpu().numpy())
        train_images.append(xb.numpy())
        if sum(b.shape[0] for b in train_latents) >= 10000:
            break
train_latents = np.concatenate(train_latents, axis=0)[:10000]
train_images = np.concatenate(train_images, axis=0)[:10000]

# Строим индекс
knn = NearestNeighbors(n_neighbors=6, algorithm="ball_tree", metric="euclidean")
knn.fit(train_latents)
print(f"KNN-индекс на {len(train_latents)} train-латентах построен.")

# Берём 4 query-картинки из test
test_xb, _ = next(iter(test_loader))
queries = test_xb[:4].to(device)
with torch.no_grad():
    q_mu, _ = vae.encode(queries)
q_mu_np = q_mu.cpu().numpy()

dists, idxs = knn.kneighbors(q_mu_np)

# Визуализация: query + 5 ближайших
fig, axes = plt.subplots(4, 6, figsize=(12, 8))
for i in range(4):
    axes[i, 0].imshow(queries[i, 0].cpu().numpy(), cmap="gray")
    axes[i, 0].set_title("query")
    axes[i, 0].axis("off")
    for j in range(5):
        axes[i, j + 1].imshow(train_images[idxs[i, j], 0], cmap="gray")
        axes[i, j + 1].set_title(f"d={dists[i, j]:.2f}")
        axes[i, j + 1].axis("off")
plt.tight_layout()
plt.show()
print("Каждая строка — query слева, 5 ближайших по латенту справа.")

## 6. Denoising Autoencoder (DAE) — бонус

**Идея.** На вход — зашумлённая картинка $\tilde x = x + n$, target — чистая $x$. Сеть учится выделять сигнал из шума:

$$\mathcal{L}_{DAE} = \mathbb{E}_{x, n} \| x - D(E(x + n)) \|_2^2$$

Шум — гауссовский с `noise_factor=0.5`, потом `clip(0, 1)`.

### Попробуй сам

- `NOISE_FACTOR = 0.5` — увеличь до 1.0 (очень сильный шум) или уменьши до 0.2 (легкий). Сравни качество очистки.

In [ ]:
class DAE(nn.Module):
    """Простой DAE — та же FC-архитектура, что у AE."""

    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(784, 256), nn.ReLU(),
            nn.Linear(256, 64), nn.ReLU(),
        )
        self.decoder = nn.Sequential(
            nn.Linear(64, 256), nn.ReLU(),
            nn.Linear(256, 784), nn.Sigmoid(),
        )

    def forward(self, x_noisy):
        z = self.encoder(x_noisy.view(x_noisy.size(0), -1))
        return self.decoder(z).view(-1, 1, 28, 28)


NOISE_FACTOR = 0.5
EPOCHS_DAE = 3

dae = DAE().to(device)
opt = torch.optim.Adam(dae.parameters(), lr=1e-3)

for epoch in range(1, EPOCHS_DAE + 1):
    dae.train()
    running = 0.0
    for xb, _ in train_loader:
        xb = xb.to(device)
        x_noisy = (xb + NOISE_FACTOR * torch.randn_like(xb)).clamp(0, 1)
        opt.zero_grad()
        recon = dae(x_noisy)
        loss = F.mse_loss(recon, xb)
        loss.backward()
        opt.step()
        running += loss.item() * xb.size(0)
    print(f"DAE Epoch {epoch}/{EPOCHS_DAE} | train MSE = {running / len(train_set):.5f}")

# Визуализация: noisy -> denoised -> original
dae.eval()
with torch.no_grad():
    xb, _ = next(iter(test_loader))
    xb = xb[:8].to(device)
    x_noisy = (xb + NOISE_FACTOR * torch.randn_like(xb)).clamp(0, 1)
    recon = dae(x_noisy)

triple = torch.cat([x_noisy, recon, xb], dim=0)
print("DAE: верх — зашумлённая (input), середина — очищенная (output), низ — оригинал")
show_grid(triple, ncols=8, figsize=(12, 4.5))

## 7. Итоги и уроки

| Что попробовали | Что увидели |
|---|---|
| Vanilla AE на MNIST | Хорошие реконструкции, но при sampling из $\mathcal{N}(0, I)$ — мусор (латент не структурирован). |
| VAE + KL | Sampling из $\mathcal{N}(0, I)$ работает: декодер видит «правдоподобные» $z$. |
| TSNE на VAE-латентах | Кластеры по классам — VAE неявно выучил цифры без меток. |
| CVAE один-z-разные-классы | $z$ кодирует «стиль», class задан снаружи. Десять разных цифр в одном «почерке». |
| KNN на VAE-латентах | Близкие картинки по евклидову расстоянию в латенте — действительно похожие цифры. |
| DAE | Энкодер учится выделять robust features, восстанавливая чистую картинку из шума. |

### Ключевые технические моменты

1. **`reduction='sum'` в BCE и KL** — обязательно, иначе масштабы лоссов не сходятся.
2. **`logsigma = log(σ²)`** — стандартная конвенция. Std = `exp(0.5 * logsigma)`.
3. **Reparameterization trick** — `z = mu + std * eps` сохраняет дифференцируемость.
4. **CVAE concat одинаково в encoder и decoder** — `cat([x, y_oh])` на входе encoder, `cat([z, y_oh])` на входе decoder.
5. **`NearestNeighbors(algorithm='ball_tree')`** вместо устаревшего `LSHForest`.

### Что дальше

- **β-VAE** — добавь `kl_weight` (в коде VAE замени `bce + kl` на `bce + beta * kl`) и попробуй `beta=4`. Кластеры станут более disentangled (одна ось латента = один атрибут).
- **VQ-VAE** — discrete codebook вместо continuous; основа DALL·E.
- **Diffusion models** — современное SOTA для генерации картинок (Stable Diffusion и т.д.).

См. полный конспект в `SUMMARY.md`.
